# 📥 Download & Organize TACO Dataset

### 🌟 What is the TACO Dataset?
**TACO (Trash Annotations in Context)** is an open-source image dataset containing diverse instances of litter in natural environments. It provides pixel-level segmentation masks and bounding boxes for various classes of garbage.

### 🎯 Why is it used?
Litter detection in real-world scenarios is complex due to occlusion, varying lighting, and diverse backgrounds. TACO provides a robust foundational dataset to train generalized models for detecting plastic and trash.

### 📊 Dataset Size
It contains roughly ~1500 high-resolution images featuring multiple annotated instances of litter across up to 60 categories.

### 📝 COCO Annotation Format
The dataset natively uses the **COCO (Common Objects in Context)** format, storing all annotations (bounding boxes, segmentation masks, categories) in a single JSON file.

### 🚀 Usage in PlasticSense AI
This notebook downloads the raw dataset, extracts the images, flags missing/broken Flickr links, validates the integrity, and prepares it cleanly. Later, these COCO annotations will be converted to YOLO format to train our PlasticSense AI YOLOv11 model.

## Step 2: Mount Google Drive
Mounting the drive ensures we don't lose the downloaded data if the Colab session disconnects.

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 3: Create Directory Structure
We use `pathlib` to strictly define and construct the required `PlasticSense_AI` folder tree safely.

In [9]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/PlasticSense_AI')

DIRS = {
    "taco_raw": PROJECT_ROOT / "datasets/taco/raw",
    "taco_images": PROJECT_ROOT / "datasets/taco/images",
    "taco_annotations": PROJECT_ROOT / "datasets/taco/annotations",
    "taco_reports": PROJECT_ROOT / "datasets/taco/reports",
    "notebooks": PROJECT_ROOT / "notebooks",
    "models": PROJECT_ROOT / "models",
    "configs": PROJECT_ROOT / "configs",
    "logs": PROJECT_ROOT / "logs"
}

for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"✔ Verified Directory: {path}")

✔ Verified Directory: /content/drive/MyDrive/PlasticSense_AI/datasets/taco/raw
✔ Verified Directory: /content/drive/MyDrive/PlasticSense_AI/datasets/taco/images
✔ Verified Directory: /content/drive/MyDrive/PlasticSense_AI/datasets/taco/annotations
✔ Verified Directory: /content/drive/MyDrive/PlasticSense_AI/datasets/taco/reports
✔ Verified Directory: /content/drive/MyDrive/PlasticSense_AI/notebooks
✔ Verified Directory: /content/drive/MyDrive/PlasticSense_AI/models
✔ Verified Directory: /content/drive/MyDrive/PlasticSense_AI/configs
✔ Verified Directory: /content/drive/MyDrive/PlasticSense_AI/logs


## Step 4: Install Required Libraries
Installing all necessary computer vision and visualization dependencies.

In [10]:
!pip install -q opencv-python pillow numpy pandas matplotlib tqdm pycocotools rich

## Step 5: Clone the Official TACO Repository
We clone the official repository into our `taco/raw/` folder.

In [18]:
import subprocess

TACO_REPO_PATH = DIRS["taco_raw"] / "TACO"

if not TACO_REPO_PATH.exists():
    print("Cloning official TACO repository...")
    !git clone https://github.com/pedropro/TACO.git "{TACO_REPO_PATH}"
else:
    print("✔ TACO repository exists. Pulling latest changes...")
    !cd "{TACO_REPO_PATH}" && git pull

if TACO_REPO_PATH.exists():
    print("\n--- Repository Metadata ---")
    !cd "{TACO_REPO_PATH}" && echo "Branch: " $(git rev-parse --abbrev-ref HEAD)
    !cd "{TACO_REPO_PATH}" && echo "Commit: " $(git rev-parse HEAD)

    # Install repo-specific requirements if they exist
    req_file = TACO_REPO_PATH / "requirements.txt"
    if req_file.exists():
        print("Installing TACO repository dependencies...")
        !pip install -q -r "{req_file}"

    print("\n✔ Verification successful.")

Cloning official TACO repository...
Cloning into '/content/drive/MyDrive/PlasticSense_AI/datasets/taco/raw/TACO'...
remote: Enumerating objects: 740, done.
remote: Counting objects: 100% (300/300), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 740 (delta 289), reused 288 (delta 288), pack-reused 440 (from 1)
Receiving objects: 100% (740/740), 107.77 MiB | 11.97 MiB/s, done.
Resolving deltas: 100% (499/499), done.
Updating files: 100% (25/25), done.

--- Repository Metadata ---
Branch:  master
Commit:  29de1a9ba05a647b83a90f18d7772e20bb23d846
Installing TACO repository dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 77.5 MB/s eta 0

## Step 6: Execute the Official Download Script
The official TACO `download.py` parses the annotations and fetches the raw images from Flickr. It natively skips broken links.

In [19]:
print("Preparing environment for TACO download script...")

# The download.py script expects a 'data' folder in the repo root to save images
taco_data_dir = TACO_REPO_PATH / "data"
taco_data_dir.mkdir(parents=True, exist_ok=True)

print("Starting official TACO download script... This may take several minutes.")
# Running with !cd and python to ensure local relative paths in download.py work correctly
!cd "{TACO_REPO_PATH}" && python download.py

print("\n✔ Download script finished executing.")

Preparing environment for TACO download script...
Starting official TACO download script... This may take several minutes.
Note. If for any reason the connection is broken. Just call me again and I will start where I left.
Finished

✔ Download script finished executing.


## Step 7: Reorganize and Copy Dataset
We extract the actual images and the `annotations.json` from the repository and place them cleanly in our curated structure, leaving the repo untouched.

In [21]:
import shutil
from tqdm.auto import tqdm

def reorganize_taco(repo_path: Path, img_dest: Path, ann_dest: Path):
    # 1. Ensure destination directories exist
    img_dest.mkdir(parents=True, exist_ok=True)
    ann_dest.mkdir(parents=True, exist_ok=True)

    # 2. Identify source annotation file
    # The script usually puts it in TACO/data/annotations.json
    source_ann = repo_path / 'data' / 'annotations.json'
    if not source_ann.exists():
        # Fallback to repo root
        source_ann = repo_path / 'annotations.json'

    if source_ann.exists():
        shutil.copy2(source_ann, ann_dest / 'annotations.json')
        print(f"✔ Copied annotations.json to {ann_dest}")
    else:
        print("✖ Error: annotations.json not found. The download script might have failed to create it.")
        return

    # 3. Copy Images
    source_data_dir = repo_path / 'data'
    all_images = list(source_data_dir.rglob('*.jpg')) + list(source_data_dir.rglob('*.jpeg')) + list(source_data_dir.rglob('*.png'))

    if not all_images:
        print("⚠ No images found in repository 'data' folder. Check if the download script encountered network errors.")
        return

    copied_count = 0
    for img_path in tqdm(all_images, desc="Organizing Images"):
        # Flatten structure: TACO/data/batch_1/00000.jpg -> taco/images/00000.jpg
        dest_path = img_dest / img_path.name
        if not dest_path.exists():
            shutil.copy2(img_path, dest_path)
        copied_count += 1

    print(f"✔ Successfully organized {copied_count} images into {img_dest}")

reorganize_taco(TACO_REPO_PATH, DIRS['taco_images'], DIRS['taco_annotations'])


✔ Copied annotations.json to /content/drive/MyDrive/PlasticSense_AI/datasets/taco/annotations


Organizing Images:   0%|          | 0/833 [00:00<?, ?it/s]

✔ Successfully organized 833 images into /content/drive/MyDrive/PlasticSense_AI/datasets/taco/images


## Step 8 & 9: Rigorous Dataset Validation
We use `pycocotools`, `PIL`, and MD5 Hashing to cross-reference the downloaded images against the annotations.

Crucially, we detect missing images (dead Flickr links), corrupted images, and generate a missing files report.

In [14]:
import json
import hashlib
import pandas as pd
from PIL import Image
from pycocotools.coco import COCO
from rich.console import Console
from rich.table import Table
import logging
import sys

console = Console()

def validate_dataset(images_dir: Path, ann_path: Path, reports_dir: Path):
    if not ann_path.exists():
        console.print("[red]Cannot validate: annotations.json missing.[/red]")
        return None, None

    console.print("[cyan]Loading COCO Annotations...[/cyan]")
    coco = COCO(ann_path)

    expected_images = coco.dataset.get('images', [])
    expected_categories = coco.dataset.get('categories', [])
    expected_annotations = coco.dataset.get('annotations', [])

    missing_images = []
    corrupted_images = []
    duplicate_images = []
    resolutions = []
    seen_hashes = set()

    total_folder_size = sum(f.stat().st_size for f in images_dir.glob('*') if f.is_file())

    console.print("[cyan]Validating individual images...[/cyan]")
    successful_images = 0
    for img_info in tqdm(expected_images, desc="Checking integrity"):
        # TACO file names in json often include the batch folder (e.g. batch_1/000001.jpg)
        # Since we flattened them, we check by the basename.
        img_name = Path(img_info['file_name']).name
        img_path = images_dir / img_name

        if not img_path.exists():
            missing_images.append({
                "image_id": img_info['id'],
                "file_name": img_info['file_name'],
                "flickr_url": img_info.get('flickr_url', 'N/A')
            })
            continue

        # Check Corruption
        try:
            with Image.open(img_path) as im:
                im.verify()
                resolutions.append(f"{im.width}x{im.height}")
        except Exception:
            corrupted_images.append(img_name)
            continue

        # Check Duplicate Hash
        hasher = hashlib.md5()
        with open(img_path, 'rb') as f:
            hasher.update(f.read())
        img_hash = hasher.hexdigest()

        if img_hash in seen_hashes:
            duplicate_images.append(img_name)
        else:
            seen_hashes.add(img_hash)
            successful_images += 1

    # Save Missing Images Report
    df_missing = pd.DataFrame(missing_images)
    if not df_missing.empty:
        df_missing.to_csv(reports_dir / "missing_images.csv", index=False)
        console.print(f"[yellow]Logged {len(missing_images)} missing images to missing_images.csv[/yellow]")

    # Display Validation Results
    table = Table(title="Dataset Integrity Report", style="cyan")
    table.add_column("Metric", justify="left")
    table.add_column("Value", justify="right")

    table.add_row("Total Expected Images", str(len(expected_images)))
    table.add_row("Successfully Validated", f"[green]{successful_images}[/green]")
    table.add_row("Missing Images (Dead Links)", f"[red]{len(missing_images)}[/red]")
    table.add_row("Broken/Corrupted Images", str(len(corrupted_images)))
    table.add_row("Duplicate Files", str(len(duplicate_images)))
    table.add_row("Total Categories", str(len(expected_categories)))
    table.add_row("Total Annotations", str(len(expected_annotations)))
    table.add_row("Folder Size (MB)", f"{total_folder_size / (1024*1024):.2f} MB")

    console.print(table)

    stats = {
        "Total Expected": len(expected_images),
        "Successfully Downloaded": successful_images,
        "Missing": len(missing_images),
        "Categories": len(expected_categories),
        "Annotations": len(expected_annotations),
        "Corrupted": len(corrupted_images),
        "Size_MB": round(total_folder_size / (1024*1024), 2),
        "Resolutions": resolutions,
        "Categories_List": [c['name'] for c in expected_categories]
    }
    return stats, coco

stats, coco_obj = validate_dataset(DIRS["taco_images"], DIRS["taco_annotations"] / "annotations.json", DIRS["taco_reports"])

Cannot validate: annotations.json missing.

## Step 10: Visualizations & Statistics
Displays a 9-image random sample grid, and provides insights into image resolutions and categories.

In [15]:
import matplotlib.pyplot as plt
import random
from collections import Counter

def visualize_dataset(stats, images_dir: Path):
    console.print("\n[bold magenta]--- Category List ---[/bold magenta]")
    console.print(", ".join(stats["Categories_List"][:15]) + " ... (truncated)")

    console.print("\n[bold magenta]--- Resolution Statistics ---[/bold magenta]")
    res_counts = Counter(stats["Resolutions"])
    most_common = res_counts.most_common(3)
    for res, count in most_common:
        console.print(f"- {res}px : {count} images")

    all_imgs = list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.jpeg')) + list(images_dir.glob('*.png'))
    if not all_imgs:
        return

    samples = random.sample(all_imgs, min(9, len(all_imgs)))
    plt.figure(figsize=(12, 12))
    for i, img_path in enumerate(samples):
        try:
            img = Image.open(img_path).convert('RGB')
            plt.subplot(3, 3, i+1)
            plt.imshow(img)
            plt.title(img_path.name, fontsize=10)
            plt.axis('off')
        except Exception:
            pass
    plt.tight_layout()
    plt.show()

if stats:
    visualize_dataset(stats, DIRS["taco_images"])

## Step 11: Generate Final Reports
Save `dataset_summary.json` and `dataset_summary.csv` for documentation and further pipeline steps.

In [16]:
def save_reports(stats, reports_dir: Path):
    if not stats: return

    clean_stats = {k: v for k, v in stats.items() if k not in ["Resolutions", "Categories_List"]}

    # JSON Export
    json_path = reports_dir / "dataset_summary.json"
    with open(json_path, 'w') as f:
        json.dump(clean_stats, f, indent=4)

    # CSV Export
    csv_path = reports_dir / "dataset_summary.csv"
    df = pd.DataFrame([clean_stats])
    df.to_csv(csv_path, index=False)

    console.print(f"[green]✔ Reports saved to {reports_dir}[/green]")

save_reports(stats, DIRS["taco_reports"])

## Step 12: Print Final Status
Render the final `PlasticSense_AI` project tree and declare readiness for the YOLO formatting phase.

In [17]:
from rich.tree import Tree
import os

def render_tree(dir_path: Path, tree: Tree):
    for path in sorted(dir_path.iterdir()):
        if path.name.startswith('.') or path.name == 'raw': # hide hidden files and raw repo dump for cleanliness
            continue
        if path.is_dir():
            branch = tree.add(f"[bold blue]{path.name}[/bold blue]")
            render_tree(path, branch)
        else:
            tree.add(path.name)

project_tree = Tree("[bold cyan]PlasticSense_AI[/bold cyan]")
render_tree(PROJECT_ROOT, project_tree)
console.print(project_tree)

console.print("\n[bold green]✔ Download completed[/bold green]")
console.print("[bold green]✔ Dataset validated[/bold green]")
console.print("[bold magenta]✔ Ready for COCO→YOLO conversion[/bold magenta]")

PlasticSense_AI
├── configs
│   └── config.py
├── datasets
├── exports
│   ├── dataset_summary.csv
│   └── dataset_summary.json
├── logs
│   ├── dataset_download.log
│   └── project_setup.log
├── models
├── notebooks
└── results
    ├── evaluation
    ├── inference
    └── training

✔ Download completed

✔ Dataset validated

✔ Ready for COCO→YOLO conversion